# Kernel Methods & One-Class SVM — Lab Exercises (with Solutions)

**Course:** Machine Learning and Analytics (SUTD)  
**Duration:** ~10 minutes  

---

### What You Will Learn

1. Why accuracy fails on imbalanced data (the *accuracy paradox*)  
2. How to train a **One-Class SVM** — using **only normal** data  
3. How hyperparameter `nu` controls the precision/recall trade-off

### Dataset

[Credit Card Fraud Detection 2023](https://www.kaggle.com/datasets/nelgiriyewithana/credit-card-fraud-detection-dataset-2023) — 568k real European transactions, anonymised into features `V1`–`V28` + `Amount`. Labels: `0 = normal`, `1 = fraud`.

---
## Setup — Run Once

In [ ]:
# Install gdown if needed, then download the dataset (no API key required)
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown', '-q'])

import gdown
DATA_FILE = 'creditcard_2023.csv'
if not os.path.exists(DATA_FILE):
    print('Downloading dataset (~310 MB)...')
    gdown.download(id='1wohtgfauFBOqdIiyBaOGvt70qnWUhNnI', output=DATA_FILE, quiet=False)
else:
    print('Dataset already downloaded.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics

sns.set_style('whitegrid')
print('Ready!')

In [ ]:
# Load & prepare a small imbalanced dataset (5% fraud)
df = pd.read_csv(DATA_FILE, index_col='id')

np.random.seed(42)
normal_df = df[df['Class'] == 0]
fraud_df  = df[df['Class'] == 1]

n_normal = 5_000
n_fraud  = int(n_normal * 0.05 / 0.95)  # ~263 (~5% fraud rate)

df_work = pd.concat([
    normal_df.sample(n_normal, random_state=42),
    fraud_df.sample(n_fraud,   random_state=42)
]).sample(frac=1, random_state=42)

X = df_work.drop('Class', axis=1).values
y = df_work['Class'].values

print(f'Dataset: {len(df_work):,} rows | Normal: {(y==0).sum()} | Fraud: {(y==1).sum()} | Fraud rate: {y.mean():.1%}')

---

## Exercise 1 - The Accuracy Paradox

When only 5% of transactions are fraud, a model that **always predicts "normal"** still scores ~95% accuracy — but catches **zero fraud**.

**Task:** Compute Accuracy, Precision, Recall, and F1 for a dummy baseline that predicts all samples as normal.

> **Hint:** `np.zeros_like(y_test)` returns an array of zeros with the same length as `y_test`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Logistic Regression (uses fraud labels) ──────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
print('Logistic Regression:')
print(f'  Accuracy={metrics.accuracy_score(y_test, y_pred_lr):.3f}  '
      f'Precision={metrics.precision_score(y_test, y_pred_lr, zero_division=0):.3f}  '
      f'Recall={metrics.recall_score(y_test, y_pred_lr, zero_division=0):.3f}  '
      f'F1={metrics.f1_score(y_test, y_pred_lr, zero_division=0):.3f}')

print()

# ── SOLUTION: Dummy baseline ──────────────────────────────────
y_pred_dummy = np.zeros_like(y_test)   # predict everything as 0 (normal)

print('Dummy Baseline (predict all normal):')
print(f'  Accuracy={metrics.accuracy_score(y_test, y_pred_dummy):.3f}  '
      f'Precision={metrics.precision_score(y_test, y_pred_dummy, zero_division=0):.3f}  '
      f'Recall={metrics.recall_score(y_test, y_pred_dummy, zero_division=0):.3f}  '
      f'F1={metrics.f1_score(y_test, y_pred_dummy, zero_division=0):.3f}')

# ANSWER: Recall drops to 0 — the dummy catches zero fraud cases.
# High accuracy is misleading because 95% of data is normal; simply predicting
# "always normal" gets 95% accuracy for free. Recall = TP / (TP+FN) = 0 means
# every single fraud transaction is missed — useless for a bank.

---

## Background: Evaluation Metrics for Anomaly Detection

Standard accuracy is misleading on imbalanced data. These are the metrics that actually matter:

| Metric | Formula | What it measures |
|--------|---------|------------------|
| **Precision** | TP / (TP + FP) | Of all transactions flagged as fraud, how many actually were? |
| **Recall (Sensitivity)** | TP / (TP + FN) | Of all actual fraud cases, how many did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Area under ROC curve | Probability that the model scores a fraud higher than a normal — uses continuous scores |
| **PR-AUC** | Area under Precision-Recall curve | Better than ROC-AUC for **highly imbalanced** data |

> **Note on One-Class SVM output:** The model returns `+1` (normal) or `−1` (anomaly). We flip this to match our labels (`1 = fraud, 0 = normal`). The `decision_function` returns a continuous score — more negative means more anomalous. We **negate** it so *higher score = more anomalous*, which is what sklearn's ROC/PR functions expect.

---

## Exercise 2 - Train & Evaluate a One-Class SVM

Unlike Exercise 1, we train using **only normal transactions** — no fraud labels needed during training.

**One-Class SVM outputs:** `+1 = normal`, `−1 = anomaly`

**Task:** Fill in the three TODOs to complete the training and evaluation pipeline.

In [ ]:
# Training set = only normal transactions (no fraud labels used!)
X_train_normal = X_train_s[y_train == 0]
print(f'Training on {len(X_train_normal):,} normal samples only (no fraud labels)')

# ── SOLUTION 1: Create and fit the One-Class SVM ─────────────
model = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
model.fit(X_train_normal)

# ── SOLUTION 2: Predict on test set, convert to 0/1 labels ───
# One-Class SVM: -1 = anomaly → map to 1 (fraud)
#               +1 = normal  → map to 0 (normal)
y_pred_svm = (model.predict(X_test_s) == -1).astype(int)

# ── SOLUTION 3: Compute Precision, Recall, F1, ROC-AUC ───────
anomaly_scores = -model.decision_function(X_test_s)   # higher = more anomalous

print('One-Class SVM:')
print(f'  Precision={metrics.precision_score(y_test, y_pred_svm, zero_division=0):.3f}  '
      f'Recall={metrics.recall_score(y_test, y_pred_svm, zero_division=0):.3f}  '
      f'F1={metrics.f1_score(y_test, y_pred_svm, zero_division=0):.3f}  '
      f'ROC-AUC={metrics.roc_auc_score(y_test, anomaly_scores):.3f}')

# ANSWER: The model was trained only on normal data but still detects fraud because
# fraud transactions lie OUTSIDE the learned boundary of "normal" behaviour.
# The RBF kernel maps data to a high-dimensional space where a hyperplane separates
# the normal cluster from the origin. Anything with a negative decision score
# (outside the boundary) is flagged as anomalous — regardless of its label.

---

## Exercise 3 - Effect of `nu` on Precision vs Recall

`nu` ∈ (0, 1] controls the **tightness** of the boundary:
- **Small `nu`** → tight boundary → few anomalies flagged → **high Precision, low Recall**  
- **Large `nu`** → loose boundary → many anomalies flagged → **low Precision, high Recall**

**Task:** Complete the loop to fill in the comparison table for `nu` ∈ `[0.05, 0.1, 0.15, 0.2, 0.3]`.

Then answer: **which `nu` gives the best F1?**

In [ ]:
nu_values = [0.05, 0.1, 0.15, 0.2, 0.3]
rows = []

for nu in nu_values:
    # SOLUTION: Create, fit, and evaluate One-Class SVM for each nu
    m = OneClassSVM(kernel='rbf', nu=nu, gamma='scale')
    m.fit(X_train_normal)
    y_pred_nu = (m.predict(X_test_s) == -1).astype(int)
    scores_nu = -m.decision_function(X_test_s)
    rows.append({
        'nu':        nu,
        'Precision': metrics.precision_score(y_test, y_pred_nu, zero_division=0),
        'Recall':    metrics.recall_score(y_test, y_pred_nu, zero_division=0),
        'F1':        metrics.f1_score(y_test, y_pred_nu, zero_division=0),
        'ROC-AUC':   metrics.roc_auc_score(y_test, scores_nu),
    })

df_results = pd.DataFrame(rows).set_index('nu').round(3)
display(df_results)

best_nu = df_results['F1'].idxmax()
print(f'\nBest F1 achieved at nu = {best_nu}')

# ANSWER Q1: As nu increases, Recall increases because the boundary becomes looser —
# the model flags more points as anomalies, catching more true fraud cases.
# However, it also flags more normal transactions (false positives), so Precision drops.

# ANSWER Q2: The best F1 is usually around nu=0.1–0.15 (balanced boundary).
# For a bank, you might choose a HIGHER nu (e.g., 0.2–0.3) to maximise Recall —
# missing real fraud is more costly than occasionally blocking a legit transaction.

---

## Reflection & Open Questions

### When to use One-Class SVM?

One-Class SVM is a strong choice when:
- You have **abundant normal data** but **few or no labelled anomalies**
- The anomalies are **diverse** and hard to model directly
- The data is **moderate-dimensional** (up to a few hundred features)

### Trade-offs in Practice

| Consideration | Details |
|---------------|---------|
| **False Positives** | Flagging normal transactions as fraud causes customer friction (blocked cards, manual review costs). |
| **False Negatives** | Missing actual fraud leads to financial losses. |
| **Threshold tuning** | The `decision_function` returns continuous scores; choosing where to draw the line is a **business decision**, not just a statistical one. |

### Limitations of One-Class SVM

- **Scalability:** Training complexity is roughly $O(n^2)$ to $O(n^3)$ — prohibitive for very large datasets (> 100k samples).
- **Sensitivity to hyperparameters:** `nu` and `gamma` can dramatically change results. Cross-validation is tricky without labelled anomalies.
- **Feature engineering matters:** Like all kernel methods, garbage in = garbage out.

### Extensions & Alternatives

| Method | Key Idea | When to Prefer |
|--------|----------|----------------|
| **Isolation Forest** | Anomalies are easier to isolate (fewer random splits needed). | Large datasets, high dimensions |
| **Local Outlier Factor (LOF)** | Compares local density of a point to its neighbours. | When anomalies have different local densities |
| **Autoencoders** | Learn to reconstruct normal data; anomalies have high reconstruction error. | Very high-dimensional data (images, sequences) |
| **Gaussian Mixture Models** | Fit a probabilistic model; low-probability points are anomalies. | When normal data has multiple clear clusters |

---

## Key Takeaways

| Concept | Summary |
|---------|--------|
| **Accuracy Paradox** | On imbalanced data, 95% accuracy means nothing if Recall = 0. Always use **Precision, Recall, F1, ROC-AUC**. |
| **One-Class SVM** | Trained on normal data only — no fraud labels needed. Learns a tight boundary; anything outside = anomaly. |
| **`nu` parameter** | Small → tight boundary (high precision, low recall). Large → loose boundary (low precision, high recall). |
| **`gamma` parameter** | Large γ → complex local boundary (overfit risk). Small γ → smooth boundary (underfit risk). |
| **Evaluation** | Prefer **PR-AUC** over ROC-AUC for highly imbalanced data — it is more sensitive to minority-class performance. |
| **When to use** | Abundant normal data, few/no labelled anomalies, diverse anomaly types, moderate dimensions. |
| **Limitation** | O(n²–n³) training — slow for > 100k samples. Consider Isolation Forest for speed. |

---

## References

1. **Schölkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J., & Williamson, R. C.** (2001). *Estimating the Support of a High-Dimensional Distribution.* Neural Computation, 13(7), 1443–1471.  
   *The foundational paper introducing the One-Class SVM formulation.*
2. **scikit-learn documentation** - [OneClassSVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.OneClassSVM.html)
3. **scikit-learn User Guide** - [Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
4. **Dataset** - [Credit Card Fraud Detection Dataset 2023](https://www.kaggle.com/datasets/nelgiriyewithana/credit-card-fraud-detection-dataset-2023), Kaggle.
5. **Course materials** - Machine Learning and Analytics, SUTD.

---
*End of Lab. Good luck!*